# Feature Selection using MRMR

In [50]:
import core.constants as c
from core.utils import save_df_as_table_image
import os

import warnings

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from feature_engine.selection import SmartCorrelatedSelection
from sklearn.base import BaseEstimator, TransformerMixin

from feature_engine.selection import MRMR

import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [33]:
data = pd.read_csv(c.RICKD_FEATURE_SELECTION_PART_2_DATA_FILE, index_col=0)
display(data.head())

data.info(verbose=True)

,age,height,weight,gender,is_injured,dom_leg_ankle_df_peak_angle,dom_leg_diff_ankle_df_peak_angle,dom_leg_ankle_eve_excursion,dom_leg_diff_ankle_eve_excursion,dom_leg_ankle_eve_peak_angle,...,dom_leg_stride_length,dom_leg_diff_stride_length,dom_leg_stride_rate,dom_leg_diff_stride_rate,dom_leg_supination_timing,dom_leg_diff_supination_timing,dom_leg_swing_time,dom_leg_diff_swing_time,dom_leg_vertical_oscillation,dom_leg_diff_vertical_oscillation
id,,,,,,,,,,,,,,,,,,,,,
100001_20110531T161051,47.0,172.0,61.9,female,True,24.591474,1.552075,-6.336418,1.187877,-1.522192,...,1.891817,0.000000,78.947368,0.000000,61,-3,0.435,0.0150,96.433793,3.847659
100002_20110601T140505,37.0,173.4,70.6,male,True,24.830009,-1.873425,-4.949355,3.761674,-10.091984,...,2.001175,0.000000,81.632653,0.000000,52,23,0.445,0.0250,86.521432,-8.424086
100003_20110601T095930,51.0,186.0,86.5,male,True,25.603645,1.020660,-9.918405,2.724571,-9.056648,...,2.241927,0.000000,78.947368,0.000000,46,-14,0.445,-0.0025,76.611379,-6.069608
100004_20110203T120721,35.0,175.6,59.0,male,True,24.519444,-1.488008,-9.005590,-1.513753,-3.335102,...,1.962250,0.000000,82.191781,0.000000,44,-8,0.440,-0.0200,92.593339,9.320156
100004_20140929T102035,39.0,175.0,61.0,male,False,24.876986,-1.453904,-12.937424,-4.962587,-11.353953,...,2.108590,-0.014643,83.333333,0.574713,43,-4,0.420,-0.0100,87.481400,12.389611


<class 'pandas.core.frame.DataFrame'>
Index: 1441 entries, 100001_20110531T161051 to 201225_20140515T133244
Data columns (total 84 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   age                                    1441 non-null   float64
 1   height                                 1441 non-null   float64
 2   weight                                 1441 non-null   float64
 3   gender                                 1441 non-null   object 
 4   is_injured                             1441 non-null   bool   
 5   dom_leg_ankle_df_peak_angle            1441 non-null   float64
 6   dom_leg_diff_ankle_df_peak_angle       1441 non-null   float64
 7   dom_leg_ankle_eve_excursion            1441 non-null   float64
 8   dom_leg_diff_ankle_eve_excursion       1441 non-null   float64
 9   dom_leg_ankle_eve_peak_angle           1441 non-null   float64
 10  dom_leg_diff_ankle_eve_peak_angle     

In [ ]:
y_label = "is_injured"

X = data.drop(columns=[y_label])
y = data[y_label]

x_cols = X.columns

cat_cols = [
    "gender",
]
num_cols = [c for c in X.columns if c not in cat_cols]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ]
)

# Pipeline
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    #("MRMR", MRMR(method="RFCQ", discrete_features=cat_cols, random_state=RANDOM_STATE))
])

X_preprocessed = pipeline.fit_transform(X, y)
X_preprocessed_df = pd.DataFrame(X_preprocessed, columns=x_cols, index=X.index)
X_preprocessed_df


,age,height,weight,gender,dom_leg_ankle_df_peak_angle,dom_leg_diff_ankle_df_peak_angle,dom_leg_ankle_eve_excursion,dom_leg_diff_ankle_eve_excursion,dom_leg_ankle_eve_peak_angle,dom_leg_diff_ankle_eve_peak_angle,...,dom_leg_stride_length,dom_leg_diff_stride_length,dom_leg_stride_rate,dom_leg_diff_stride_rate,dom_leg_supination_timing,dom_leg_diff_supination_timing,dom_leg_swing_time,dom_leg_diff_swing_time,dom_leg_vertical_oscillation,dom_leg_diff_vertical_oscillation
id,,,,,,,,,,,,,,,,,,,,,
100001_20110531T161051,0.744600,-0.058986,-0.641070,1.135059,0.645463,1.088779,0.475144,1.424680,0.506794,1.345865,...,0.034516,-0.804811,-0.026732,0.510085,-0.310250,0.221441,1.091805,0.540432,0.346772,0.0
100002_20110601T140505,-0.130721,0.090330,-0.003485,1.203886,-0.784673,1.422988,1.359352,-0.890294,-0.126587,0.765725,...,0.034516,-0.282344,-0.026732,-0.378709,2.369282,0.480694,1.797845,-0.059281,-1.030256,1.0
100003_20110601T095930,1.094728,1.434171,1.161757,1.427113,0.423599,0.225710,1.003063,-0.610616,-0.008311,-0.116054,...,0.034516,-0.804811,-0.026732,-0.971238,-1.443897,0.480694,-0.143764,-0.658855,-0.766057,1.0
100004_20110203T120721,-0.305785,0.324969,-0.853598,1.114275,-0.623762,0.445650,-0.452980,0.934955,0.216169,-0.272763,...,0.034516,-0.173556,-0.026732,-1.168747,-0.825544,0.351067,-1.379333,0.308078,0.960847,1.0
100004_20140929T102035,0.044343,0.260976,-0.707027,1.217441,-0.609524,-0.501714,-1.637799,-1.231192,-0.147565,-1.030734,...,-2.380927,0.048552,2.235014,-1.267502,-0.413308,-0.167438,-0.673294,-0.001202,1.305274,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201100_20150409T155915,1.182261,-0.272294,0.685400,-1.504158,-0.782999,-0.585586,-0.746858,-0.762015,1.545030,-0.129628,...,0.034516,-1.294000,-0.026732,1.201369,3.915166,0.999199,0.385766,-0.047836,1.256745,1.0
201101_20150413T143152,-1.531235,-1.125527,-0.377241,-1.889810,-0.890220,-1.863008,-0.474742,-3.217498,-4.816930,-2.326411,...,0.034516,1.002311,-0.026732,-0.576218,-2.062251,-0.426690,0.209256,-1.219550,-1.063721,1.0
201223_20140415T132359,-0.830978,-0.165640,0.099115,-0.473019,-6.584194,-0.740890,-3.357463,0.350933,0.327934,0.076932,...,2.407312,-0.600004,-2.139719,2.188917,-1.856133,1.712144,7.446161,1.308390,1.971056,1.0


In [ ]:
# Balanced parameter grid for your specific case
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 7, 10, None],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [2, 4, 8],
    "max_features": ["sqrt", "log2", 0.5],
    "bootstrap": [True, False]
}

# Relevance: Random Forest; Redundancy: Pearson Correlation; Comparison: Ratio
sel = MRMR(
    method="RFCQ",
    scoring="roc_auc",
    param_grid = param_grid,
    cv=5,
    regression=False
)
sel.fit(X, y)

pd.Series(sel.relevance_, index=sel.variables_).sort_values(
    ascending=False).plot.bar(figsize=(15, 4))
plt.title("Relevance")
plt.show()

In [ ]:
pd.Series(sel.relevance_, index=sel.variables_).sort_values(
    ascending=False).plot.bar(figsize=(15, 4))
plt.title("Feature Relevance")
plt.show()

In [ ]:
Xtr = sel.transform(X)
print(Xtr.head())